# Math 07: Calculus — Differential Equations**Learning Objectives:**1. Solve separable differential equations analytically2. Visualize differential equations using Slope Fields3. Implement Euler's Method to approximate solutions computationally**Prerequisites:** Math 02 (Derivatives)**Estimated time:** 90 minutes**Textbook:** *Justin Skycak — Calculus* §3.1-3.2

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_07_calculus_differential_equations",    "level": 1,    "title": "Math 07: Calculus \u2014 Differential Equations",    "prerequisites": [        "Math 02 (Derivatives)"    ],    "skills_taught": [        "lesson.math_07_calculus_differential_equations"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "01_introductory_coding.ipynb"}SKILLS = [    {        "id": "lesson.math_07_calculus_differential_equations",        "title": "Lesson Math 07 Calculus Differential Equations",        "notebook": "math_07_calculus_differential_equations.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "math_07_calculus_differential_equations.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Math 02 (Derivatives).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.math_07_calculus_differential_equations', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q sympy matplotlib plotly ipywidgetsimport sympy as sp; sp.init_printing()import numpy as np, matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### What is a Differential Equation?> 📖 *From Calculus §3.1:* In differential equations, we are given an equation in terms of the derivative(s) of some function, and we need to solve for the function that makes the equation true.For example, if $\frac{dy}{dx} = 2x$, the solution is $y = x^2 + C$.### Separation of VariablesThe simplest differential equations can be solved by **separation of variables**, where we move all $y$ terms to one side and all $x$ terms to the other, then integrate.**Example:** Solve $\frac{dy}{dx} = \frac{2x}{y}$1. Multiply both sides by $y \, dx$: $\quad y \, dy = 2x \, dx$2. Integrate both sides: $\quad \int y \, dy = \int 2x \, dx$3. Result: $\quad \frac{1}{2}y^2 = x^2 + C$4. Solve for $y$: $\quad y = \pm \sqrt{2x^2 + C}$

In [ ]:
# Symbolic solution using SymPyimport sympy as spx = sp.Symbol('x')y = sp.Function('y')(x)C1 = sp.Symbol('C1')# Define the equation: y' = 2x / yode = sp.Eq(y.diff(x), 2*x / y)# Solve itsolution = sp.dsolve(ode, y)print('General solutions:')for sol in solution:    sp.pprint(sol)

### Slope Fields> 📖 *From Calculus §3.2:* When faced with a differential equation that we don’t know how to solve, we can sometimes still approximate the solution. If we just want to get an idea of what the solutions look like on a graph, we can construct a **slope field**.A slope field is an array of short line segments. At any point $(x,y)$, the segment's slope is determined by $\frac{dy}{dx}$. If you drop a 'raft' into this 'river' of slopes, it traces out a specific solution curve.

### Comprehension Check ✅1. What is the separation of variables for $\frac{dy}{dx} = x^2 y$?2. What does the constant $C$ represent geometrically in a slope field?<details><summary>💡 Explanation after a retrieval attempt</summary>1. $\frac{1}{y} dy = x^2 dx$2. $C$ selects which specific 'current' (solution curve) you follow in the slope field. It is determined by your starting point (Initial Condition).</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Solve by HandSolve $\frac{dy}{dx} = -2xy$ with the initial condition $y(0) = 1$.Do this on paper, then check your answer by taking the derivative of your result computationally.

In [ ]:
def verify_solution():    import sympy as sp    x = sp.Symbol('x')        # Replace 'pass' with your symbolic solution expression for y(x)    # Example: y_sol = sp.exp(x)     # YOUR CODE HERE    y_sol = None         if y_sol is not None:        dy_dx = sp.diff(y_sol, x)        print(f"Your y(x) = {y_sol}")        print(f"Derivative dy/dx = {dy_dx}")        print(f"Does dy/dx == -2*x*y? {sp.simplify(dy_dx - (-2*x*y_sol)) == 0}")        print(f"Does y(0) == 1? {y_sol.subs(x, 0) == 1}")verify_solution()

---## Phase 1 — Algorithm Construction### Step 1: Visualizing a Slope FieldLet's write a function to plot the slope field for any differential equation $\frac{dy}{dx} = f(x, y)$.

In [ ]:
def plot_slope_field(f, x_range=(-3, 3), y_range=(-3, 3), density=20):    '''Plot the slope field for dy/dx = f(x, y).'''    import numpy as np, matplotlib.pyplot as plt        xs = np.linspace(x_range[0], x_range[1], density)    ys = np.linspace(y_range[0], y_range[1], density)    X, Y = np.meshgrid(xs, ys)        # Evaluate the slope at each grid point    # YOUR CODE HERE to compute dy and dx components for quiver plot    # U = ... (x-component of the vector)    # V = ... (y-component of the vector)        # Example visualization setup:    # plt.quiver(X, Y, U, V, color='blue', alpha=0.5)    pass# Test with dy/dx = -x/y (circles)# plot_slope_field(lambda x, y: -x/y if y != 0 else 0)# print('Step 1 passed ✅')

### Step 2: Euler's Method (Numerical Approximation)> 📖 *From Calculus §3.2:* Euler's method uses the slope at the current point to take a small step forward, approximating the next point on the curve.$y_{n+1} = y_n + \Delta x \cdot f(x_n, y_n)$Write a function to perform Euler's Method.

In [ ]:
def euler_method(f, x0, y0, dx, num_steps):    '''Approximate the solution to dy/dx = f(x,y) starting at (x0, y0).'''    xs, ys = [x0], [y0]    # YOUR CODE HERE    return xs, ys# Test with dy/dx = y (Exponential growth), starting at (0, 1)# xs, ys = euler_method(lambda x, y: y, 0, 1, 0.1, 10)# assert abs(ys[-1] - 2.5937) < 1e-2  # Approx e^1# print('Step 2 passed ✅')

### Step 3: Overlaying the Solution (Interleaved Practice)Combine your slope field and Euler's method to visualize the raft in the river.

In [ ]:
def plot_euler_on_field(f, x0, y0, dx, num_steps):    '''Plot a slope field and overlay an Euler approximation curve.'''    # YOUR CODE HERE    pass# Test:# plot_euler_on_field(lambda x, y: y - x, 0, 0.5, 0.2, 15)

---## Phase 2 — Experimental Verification### Pathological Case: Euler's Method DivergenceEuler's method assumes the slope is constant over the step $\Delta x$. If the curve bends sharply, Euler's method will overshoot and "fly off" the true curve.Let's visualize this with a stiff equation: $\frac{dy}{dx} = -15y$, starting at $y(0) = 1$.

In [ ]:
import numpy as np, matplotlib.pyplot as pltdef stiff_eq(x, y): return -15 * yexact_xs = np.linspace(0, 1, 100)exact_ys = np.exp(-15 * exact_xs)# Euler with a step size that is slightly too large (dx = 0.15)# Try changing dx to 0.05 and see what happens!dx = 0.15xs, ys = [0], [1]for _ in range(int(1 / dx)):    xs.append(xs[-1] + dx)    ys.append(ys[-1] + dx * stiff_eq(xs[-2], ys[-1]))plt.figure(figsize=(8,4))plt.plot(exact_xs, exact_ys, 'k-', lw=2, label='True Solution $e^{-15x}$')plt.plot(xs, ys, 'ro-', lw=2, label=f"Euler's Method (dx={dx})")plt.title("Overshooting in Euler's Method")plt.legend(); plt.grid(alpha=0.3); plt.show()

### 🔍 Reflection1. Why did Euler's method oscillate and grow instead of decaying to 0?2. What relationship between the slope $-15$ and the step size $\Delta x$ caused this?<details><summary>💡 Explanation after a retrieval attempt</summary>The step formula is $y_{n+1} = y_n + \Delta x(-15 y_n) = y_n(1 - 15\Delta x)$.If $15\Delta x > 2$ (so $\Delta x > 0.133$), then the multiplier is less than $-1$, causing the value to flip signs and grow exponentially in magnitude instead of shrinking.</details>

---## Phase 3 — Olympiad Extension### Challenge: Runge-Kutta 4th Order (RK4)Euler's method evaluates the slope once at the beginning of the step.RK4 evaluates the slope four times (beginning, middle x2, and end) to compute a much better average slope. It is the workhorse of scientific computing.Implement RK4 and compare its accuracy against Euler on the stiff equation.

In [ ]:
def rk4_step(f, x, y, dx):    '''Take a single RK4 step.'''    # YOUR CODE HERE    pass# Test and plot RK4 vs Euler!pass

### Chalkboard DefenseWrite a formal mathematical explanation of why RK4 has local truncation error $O(\Delta x^5)$ compared to Euler's $O(\Delta x^2)$.<details><summary>💡 Hint</summary>Euler's method is essentially a first-order Taylor series approximation: $y(x+\Delta x) \approx y(x) + \Delta x y'(x)$.RK4 is designed to match the Taylor series expansion up to the 4th derivative term $\frac{(\Delta x)^4}{24}y^{(4)}(x)$.</details>---📚 **Next:** Notebook 01 (Introductory Coding) begins the Algorithms & Machine Learning track.

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.math_07_calculus_differential_equations', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='01_introductory_coding.ipynb')